In [ ]:
# === Colab Setup ===
import os, sys

try:
    import google.colab
    !rm -rf /content/repo
    !git clone https://github.com/jlattanzi4/energy-demand-forecasting.git /content/repo
    os.chdir('/content/repo')
    !pip install -q scikit-learn statsmodels prophet matplotlib seaborn
except ImportError:
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')

sys.path.insert(0, os.getcwd())

import src.data.preprocess
print('Setup OK')

# Energy Demand Forecasting — Exploratory Data Analysis

**Dataset**: PJM East (PJME) Hourly Energy Consumption  
**Source**: [Kaggle](https://www.kaggle.com/datasets/robikscube/hourly-energy-consumption)  
**Records**: ~145,000 hourly observations (2002–2018)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 12

%matplotlib inline

## 1. Load and Inspect Data

In [ ]:
from src.data.preprocess import load_data

df = load_data('data/PJME_hourly.csv')
print(f'Shape: {df.shape}')
print(f'Date range: {df.index.min()} to {df.index.max()}')
print(f'\nBasic statistics:')
df.describe()

In [ ]:
print(f"Missing values: {df['consumption_mw'].isna().sum()}")
print(f'Duplicated timestamps: {df.index.duplicated().sum()}')
print(f'\nFrequency: {pd.infer_freq(df.index)}')

## 2. Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['consumption_mw'], bins=80, color='#2196F3', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Energy Consumption (MW)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Hourly Energy Consumption')
axes[0].axvline(df['consumption_mw'].mean(), color='red', linestyle='--', label=f"Mean: {df['consumption_mw'].mean():.0f} MW")
axes[0].legend()

df_yearly = df.copy()
df_yearly['year'] = df_yearly.index.year
df_yearly.boxplot(column='consumption_mw', by='year', ax=axes[1])
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Energy Consumption (MW)')
axes[1].set_title('Consumption by Year')
plt.suptitle('')

plt.tight_layout()
plt.show()

## 3. Time Series Overview

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(df.index, df['consumption_mw'], linewidth=0.3, color='#2196F3')
axes[0].set_title('PJM East — Full Hourly Time Series')
axes[0].set_ylabel('Energy (MW)')

month_slice = df['2017-07':'2017-07']
axes[1].plot(month_slice.index, month_slice['consumption_mw'], color='#FF5722', linewidth=1)
axes[1].set_title('July 2017 — Daily/Weekly Patterns Visible')
axes[1].set_ylabel('Energy (MW)')
axes[1].set_xlabel('Date')

plt.tight_layout()
plt.show()

## 4. Seasonality Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

hourly = df.groupby(df.index.hour)['consumption_mw'].agg(['mean', 'std'])
axes[0].plot(hourly.index, hourly['mean'], color='#2196F3', linewidth=2)
axes[0].fill_between(hourly.index, hourly['mean'] - hourly['std'], hourly['mean'] + hourly['std'], alpha=0.2)
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Mean Consumption (MW)')
axes[0].set_title('Daily Seasonality')
axes[0].set_xticks(range(0, 24, 3))

dow_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
weekly = df.groupby(df.index.dayofweek)['consumption_mw'].agg(['mean', 'std'])
axes[1].bar(weekly.index, weekly['mean'], color='#4CAF50', alpha=0.8, yerr=weekly['std'], capsize=3)
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(dow_names)
axes[1].set_ylabel('Mean Consumption (MW)')
axes[1].set_title('Weekly Seasonality')

monthly = df.groupby(df.index.month)['consumption_mw'].agg(['mean', 'std'])
axes[2].bar(monthly.index, monthly['mean'], color='#FF5722', alpha=0.8, yerr=monthly['std'], capsize=3)
axes[2].set_xlabel('Month')
axes[2].set_ylabel('Mean Consumption (MW)')
axes[2].set_title('Yearly Seasonality')

plt.tight_layout()
plt.show()

## 5. Autocorrelation (ACF / PACF)

In [ ]:
sample = df['consumption_mw']['2017':'2017'].values

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_acf(sample, lags=168, ax=axes[0], title='ACF (168 lags = 1 week)')
plot_pacf(sample, lags=48, ax=axes[1], title='PACF (48 lags = 2 days)', method='ywm')
plt.tight_layout()
plt.show()

print('Key observations:')
print('  - Strong autocorrelation at lag 24 (same hour yesterday)')
print('  - Strong autocorrelation at lag 168 (same hour last week)')
print('  - This motivates our lag features: t-1, t-24, t-168')

## 6. Stationarity Test

In [ ]:
result = adfuller(df['consumption_mw'].dropna(), maxlag=168)
print('Augmented Dickey-Fuller Test:')
print(f'  Test statistic: {result[0]:.4f}')
print(f'  p-value: {result[1]:.6f}')
print(f'  Lags used: {result[2]}')
print(f'  Critical values:')
for key, val in result[4].items():
    print(f'    {key}: {val:.4f}')

if result[1] < 0.05:
    print('\n=> Series is stationary (reject null hypothesis)')
else:
    print('\n=> Series may be non-stationary (fail to reject null)')

## 7. Train/Val/Test Split

In [ ]:
from src.data.preprocess import temporal_split

train, val, test = temporal_split(df)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train.index, train['consumption_mw'], label=f'Train ({len(train):,} rows)', linewidth=0.3)
ax.plot(val.index, val['consumption_mw'], label=f'Val ({len(val):,} rows)', linewidth=0.3)
ax.plot(test.index, test['consumption_mw'], label=f'Test ({len(test):,} rows)', linewidth=0.3)
ax.axvline(pd.Timestamp('2016-01-01'), color='black', linestyle='--', alpha=0.5)
ax.axvline(pd.Timestamp('2017-01-01'), color='black', linestyle='--', alpha=0.5)
ax.set_title('Temporal Train/Val/Test Split')
ax.set_ylabel('Energy (MW)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Train: {train.index.min()} to {train.index.max()}')
print(f'Val:   {val.index.min()} to {val.index.max()}')
print(f'Test:  {test.index.min()} to {test.index.max()}')

## 8. Feature Engineering Preview

In [ ]:
from src.data.preprocess import fit_scaler, transform_data
from src.data.features import create_features, FEATURE_COLUMNS

scaler = fit_scaler(train, save_path='models/scaler.pkl')
train_scaled = transform_data(train, scaler)
train_feat = create_features(train_scaled)

print(f'Feature columns ({len(FEATURE_COLUMNS)}):')
for col in FEATURE_COLUMNS:
    print(f'  - {col}')

print(f'\nRows after feature engineering: {len(train_feat):,}')
print(f'Rows dropped (NaN from lags): {len(train_scaled) - len(train_feat):,}')

train_feat[FEATURE_COLUMNS].head(10)